<a href="https://colab.research.google.com/github/aayurchik/27_toxicity_prediction/blob/main/ml_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 49.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from joblib import Parallel, delayed

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Colab Notebooks/'
data = np.load(BASE + 'aligned_data.npz', allow_pickle=True)
X_phys = data['X_phys']
X_fp = data['X_fp']
X_m2v = data['X_m2v']
Y = data['Y']
smiles = data['smiles']

targets_df = pd.read_csv(BASE + '2_targets_only.csv')
target_cols = [c for c in targets_df.columns if c != 'smiles']

print("Loaded shapes:", X_phys.shape, X_fp.shape, X_m2v.shape, Y.shape)

def get_scaffold(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        Chem.SanitizeMol(mol)
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf)
    except:
        return None

scaffolds = [get_scaffold(s) for s in smiles]
valid_idx = [i for i, sc in enumerate(scaffolds) if sc is not None]

X_phys = X_phys[valid_idx]
X_fp = X_fp[valid_idx]
X_m2v = X_m2v[valid_idx]
Y = Y[valid_idx]
smiles = [smiles[i] for i in valid_idx]
scaffolds = [scaffolds[i] for i in valid_idx]

# формируем train/test scaffold split
scaffold_dict = {}
for i, scaf in enumerate(scaffolds):
    scaffold_dict.setdefault(scaf, []).append(i)

scaffold_sets = sorted(scaffold_dict.values(), key=lambda x: -len(x))
train_idx, test_idx = [], []
test_size = int(0.2 * len(scaffolds))

for group in scaffold_sets:
    if len(test_idx) + len(group) <= test_size:
        test_idx.extend(group)
    else:
        train_idx.extend(group)

train_idx = np.array(train_idx)
test_idx  = np.array(test_idx)

X_phys_tr, X_phys_te = X_phys[train_idx], X_phys[test_idx]
X_fp_tr, X_fp_te     = X_fp[train_idx], X_fp[test_idx]
X_m2v_tr, X_m2v_te   = X_m2v[train_idx], X_m2v[test_idx]
Y_train, Y_test      = Y[train_idx], Y[test_idx]

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

Mounted at /content/drive
Loaded shapes: (10000, 78) (10000, 2048) (10000, 300) (10000, 13)
Train: 8000, Test: 2000


In [ ]:
def pr_auc(y_true, y_prob):
    mask = ~np.isnan(y_true)
    if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
        return np.nan
    return average_precision_score(y_true[mask], y_prob[mask])

def roc_auc(y_true, y_prob):
    mask = ~np.isnan(y_true)
    if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
        return np.nan
    return roc_auc_score(y_true[mask], y_prob[mask])
def rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
def et():
    return ExtraTreesClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
def xgb():
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                         scale_pos_weight=5, eval_metric="logloss", n_jobs=-1, random_state=42)
def lgb():
    from lightgbm import LGBMClassifier
    return LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                          class_weight="balanced", n_jobs=-1, random_state=42, verbose=-1)
def logreg():
    return LogisticRegression(max_iter=1000, class_weight="balanced", solver='saga', n_jobs=-1)
def knn():
    return KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

representations = {
    "PhysChem": (X_phys_tr, X_phys_te),
    "Mol2Vec+Phys": (np.hstack([X_phys_tr, X_m2v_tr]),
                     np.hstack([X_phys_te, X_m2v_te])),
    "MorganFP": (X_fp_tr, X_fp_te),
    "ALL": (np.hstack([X_phys_tr, X_fp_tr, X_m2v_tr]),
            np.hstack([X_phys_te, X_fp_te, X_m2v_te]))}

models = [("RF", rf), ("ExtraTrees", et), ("XGB", xgb), ("LGB", lgb), ("LogReg", logreg), ("KNN", knn)]

def eval_task(task_i, model_fn, X_tr, X_te):
    y_tr = Y_train[:, task_i]
    y_te = Y_test[:, task_i]

    m_tr = ~np.isnan(y_tr)
    m_te = ~np.isnan(y_te)

    if m_tr.sum() < 20 or len(np.unique(y_tr[m_tr])) < 2:
        return np.nan, np.nan

    model = model_fn()
    model.fit(X_tr[m_tr], y_tr[m_tr])
    prob = model.predict_proba(X_te[m_te])[:, 1]

    return pr_auc(y_te[m_te], prob), roc_auc(y_te[m_te], prob)

def macro_score(model_fn, X_tr, X_te):
    results = Parallel(n_jobs=2)(
        delayed(eval_task)(t, model_fn, X_tr, X_te)
        for t in range(Y.shape[1]))
    pr_vals = [r[0] for r in results]
    roc_vals = [r[1] for r in results]
    return np.nanmean(pr_vals), np.nanmean(roc_vals)

In [ ]:
results = []

for feat_name, (X_tr, X_te) in representations.items():
    for model_name, model_fn in models:
        t0 = time.time()
        pr, roc = macro_score(model_fn, X_tr, X_te)
        t = time.time() - t0
        results.append({
            "features": feat_name,
            "model": model_name,
            "PR-AUC": pr,
            "ROC-AUC": roc,
            "time_s": round(t, 1)})
        print(f"[{feat_name:15s} | {model_name:7s}] PR-AUC={pr:.4f}  time={round(t,1)}s")

df = pd.DataFrame(results).sort_values("PR-AUC", ascending=False)
print(df.to_string(index=False))
df.to_csv(BASE + "ml_final_clean_scaffold.csv", index=False)

[PhysChem        | RF     ] PR-AUC=0.5922  time=19.3s
[PhysChem        | ExtraTrees] PR-AUC=0.5990  time=7.7s
[PhysChem        | XGB    ] PR-AUC=0.5658  time=6.3s
[PhysChem        | LGB    ] PR-AUC=0.5199  time=14.1s
[PhysChem        | LogReg ] PR-AUC=0.5626  time=15.5s
[PhysChem        | KNN    ] PR-AUC=0.5151  time=0.4s
[Mol2Vec+Phys    | RF     ] PR-AUC=0.5391  time=48.7s
[Mol2Vec+Phys    | ExtraTrees] PR-AUC=0.5356  time=13.7s
[Mol2Vec+Phys    | XGB    ] PR-AUC=0.5236  time=33.3s
[Mol2Vec+Phys    | LGB    ] PR-AUC=0.5178  time=34.5s
[Mol2Vec+Phys    | LogReg ] PR-AUC=0.5675  time=68.1s
[Mol2Vec+Phys    | KNN    ] PR-AUC=0.5135  time=0.6s
[MorganFP        | RF     ] PR-AUC=0.5556  time=27.7s
[MorganFP        | ExtraTrees] PR-AUC=0.5481  time=39.2s
[MorganFP        | XGB    ] PR-AUC=0.5293  time=12.5s
[MorganFP        | LGB    ] PR-AUC=0.5206  time=2.7s
[MorganFP        | LogReg ] PR-AUC=0.5484  time=280.3s
[MorganFP        | KNN    ] PR-AUC=0.5012  time=2.2s
[ALL             | RF   

Лучшее ExtraTrees / RF на PhysChem (0.599 / 0.592), XGB/LGB сильно отстают скорее всего дефолтные гиперпараметры плохие. KNN и LGB внизу везде. Комбинации ALL хуже чем просто PhysChem, FP добавляют шум при scaffold split

In [ ]:
# from sklearn.model_selection import ParameterGrid

# hpo_results = []

# subset_tasks = list(range(min(40, Y.shape[1])))
# def macro_score_fast(model_fn, X_tr, X_te):
#     tasks = subset_tasks if subset_tasks is not None else range(Y.shape[1])

#     results = Parallel(n_jobs=2)(
#         delayed(eval_task)(t, model_fn, X_tr, X_te)
#         for t in tasks)

#     pr_vals = [r[0] for r in results]
#     roc_vals = [r[1] for r in results]

#     return np.nanmean(pr_vals), np.nanmean(roc_vals)

#     from xgboost import XGBClassifier

# def run_xgb():

#     param_grid = {
#         "n_estimators": [300, 600],
#         "max_depth": [3, 5],
#         "learning_rate": [0.03, 0.1],
#         "subsample": [0.8],
#         "colsample_bytree": [0.8],
#         "scale_pos_weight": [1, 5]
#     }

#     for feat_name, (X_tr, X_te) in representations.items():

#         print(f"\n### XGB / {feat_name}")
#         best_pr = 0

#         for params in ParameterGrid(param_grid):

#             def model_fn(p=params):
#                 return XGBClassifier(
#                     eval_metric="logloss",
#                     n_jobs=1,
#                     random_state=42,
#                     **p
#                 )

#             t0 = time.time()
#             pr, roc = macro_score_fast(model_fn, X_tr, X_te)
#             t = round(time.time() - t0, 1)

#             print(f"{params} → PR={pr:.4f} ({t}s)")

#             hpo_results.append({
#                 "model": "XGB",
#                 "features": feat_name,
#                 "params": str(params),
#                 "PR-AUC": pr,
#                 "ROC-AUC": roc,
#                 "time_s": t})

#             if pr > best_pr:
#                 best_pr = pr
#                 best_params = params

#         print("BEST:", best_params, "PR:", round(best_pr, 4))

In [ ]:
# from lightgbm import LGBMClassifier

# def run_lgb():

#     param_grid = {
#         "n_estimators": [300, 600],
#         "num_leaves": [31, 63],
#         "learning_rate": [0.03, 0.1],
#         "subsample": [0.8],
#         "colsample_bytree": [0.8],
#         "class_weight": ["balanced"]}

#     for feat_name, (X_tr, X_te) in representations.items():

#         print(f"\n### LGB / {feat_name}")
#         best_pr = 0

#         for params in ParameterGrid(param_grid):

#             def model_fn(p=params):
#                 return LGBMClassifier(
#                     n_jobs=1,
#                     random_state=42,
#                     verbose=-1,
#                     **p)

#             t0 = time.time()
#             pr, roc = macro_score_fast(model_fn, X_tr, X_te)
#             t = round(time.time() - t0, 1)

#             print(f"{params} → PR={pr:.4f} ({t}s)")

#             hpo_results.append({
#                 "model": "LGB",
#                 "features": feat_name,
#                 "params": str(params),
#                 "PR-AUC": pr,
#                 "ROC-AUC": roc,
#                 "time_s": t})

#             if pr > best_pr:
#                 best_pr = pr
#                 best_params = params

#         print("BEST:", best_params, "PR:", round(best_pr, 4))

In [ ]:
# run_xgb()
# run_lgb()

# hpo_df = pd.DataFrame(hpo_results)

# final_df = pd.concat([
#     df.assign(stage="baseline"),
#     hpo_df.assign(stage="tuned")
# ], ignore_index=True)

# final_df = final_df.sort_values("PR-AUC", ascending=False)

# print(final_df.head(20))
# final_df.to_csv(BASE + "ml_hpo_scaffold.csv", index=False)


### XGB / PhysChem
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 1, 'subsample': 0.8} → PR=0.5654 (9.3s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 5, 'subsample': 0.8} → PR=0.5810 (1.9s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 600, 'scale_pos_weight': 1, 'subsample': 0.8} → PR=0.5605 (3.0s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 600, 'scale_pos_weight': 5, 'subsample': 0.8} → PR=0.5776 (3.4s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'n_estimators': 300, 'scale_pos_weight': 1, 'subsample': 0.8} → PR=0.5686 (4.6s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'n_estimators': 300, 'scale_pos_weight': 5, 'subsample': 0.8} → PR=0.5810 (4.4s)
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'n_estimators': 600, 'scale_pos_weight'

In [ ]:
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# def make_pca_features(X_tr, X_te, n_comp):

#     max_comp = min(X_tr.shape[0], X_tr.shape[1])
#     if n_comp > max_comp:
#         n_comp = max_comp

#     scaler = StandardScaler()
#     X_tr_scaled = scaler.fit_transform(X_tr)
#     X_te_scaled = scaler.transform(X_te)

#     pca = PCA(n_components=n_comp, random_state=42)
#     X_tr_pca = pca.fit_transform(X_tr_scaled)
#     X_te_pca = pca.transform(X_te_scaled)

#     var = pca.explained_variance_ratio_.sum()
#     print(f"PCA {n_comp:3d} | var={var:.3f}")

#     return X_tr_pca, X_te_pca


# def make_pca_auto(X_tr, X_te, var_target=0.95):

#     scaler = StandardScaler()
#     X_tr_scaled = scaler.fit_transform(X_tr)
#     X_te_scaled = scaler.transform(X_te)

#     pca = PCA(n_components=var_target, random_state=42)
#     X_tr_pca = pca.fit_transform(X_tr_scaled)
#     X_te_pca = pca.transform(X_te_scaled)

#     print(f"PCA auto | comp={pca.n_components_:3d} | var={pca.explained_variance_ratio_.sum():.3f}")

#     return X_tr_pca, X_te_pca


# def xgb_best():
#     from xgboost import XGBClassifier
#     return XGBClassifier(
#         n_estimators=300,
#         max_depth=5,
#         learning_rate=0.03,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         scale_pos_weight=5,
#         eval_metric="logloss",
#         n_jobs=-1,
#         random_state=42)

# def lgb_best():
#     from lightgbm import LGBMClassifier
#     return LGBMClassifier(
#         n_estimators=300,
#         num_leaves=63,
#         learning_rate=0.03,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         class_weight="balanced",
#         n_jobs=-1,
#         random_state=42,
#         verbose=-1
#     )


# pca_dims = [20, 50]

# pca_representations = {}

# for name, (X_tr, X_te) in representations.items():

#     print(f"\n>> {name}")

#     # фиксированные размеры
#     for dim in pca_dims:
#         X_tr_pca, X_te_pca = make_pca_features(X_tr, X_te, dim)
#         pca_representations[f"{name}_PCA{dim}"] = (X_tr_pca, X_te_pca)

#     # авто PCA
#     X_tr_pca, X_te_pca = make_pca_auto(X_tr, X_te, 0.95)
#     pca_representations[f"{name}_PCAauto"] = (X_tr_pca, X_te_pca)

# pca_results = []

# for feat_name, (X_tr, X_te) in pca_representations.items():

#     for model_name, model_fn in [("XGB", xgb_best), ("LGB", lgb_best)]:

#         t0 = time.time()
#         pr, roc = macro_score(model_fn, X_tr, X_te)
#         t = round(time.time() - t0, 1)

#         row = {
#             "features": feat_name,
#             "model": model_name,
#             "PR-AUC": round(pr, 4),
#             "ROC-AUC": round(roc, 4),
#             "time_s": t
#         }

#         pca_results.append(row)

#         print(f"[{feat_name:20s} | {model_name}] PR={pr:.4f} time={t}s")

# pca_df = pd.DataFrame(pca_results).sort_values("PR-AUC", ascending=False)

# print("\n  results")
# print(pca_df.to_string(index=False))

# pca_df.to_csv(BASE + "pca_boosting_results.csv", index=False)


>> PhysChem
PCA  20 | var=0.820
PCA  50 | var=0.990
PCA auto | comp= 37 | var=0.953

>> Mol2Vec+Phys
PCA  20 | var=0.688
PCA  50 | var=0.848
PCA auto | comp=120 | var=0.951

>> MorganFP
PCA  20 | var=0.062
PCA  50 | var=0.124
PCA auto | comp=1496 | var=0.950

>> ALL
PCA  20 | var=0.148
PCA  50 | var=0.219
PCA auto | comp=1511 | var=0.950
[PhysChem_PCA20       | XGB] PR=0.5763 time=5.1s
[PhysChem_PCA20       | LGB] PR=0.5214 time=6.5s
[PhysChem_PCA50       | XGB] PR=0.5540 time=4.4s
[PhysChem_PCA50       | LGB] PR=0.5380 time=8.8s
[PhysChem_PCAauto     | XGB] PR=0.5688 time=3.3s
[PhysChem_PCAauto     | LGB] PR=0.5285 time=5.4s
[Mol2Vec+Phys_PCA20   | XGB] PR=0.5457 time=3.3s
[Mol2Vec+Phys_PCA20   | LGB] PR=0.5191 time=5.1s
[Mol2Vec+Phys_PCA50   | XGB] PR=0.5422 time=4.4s
[Mol2Vec+Phys_PCA50   | LGB] PR=0.5142 time=9.2s
[Mol2Vec+Phys_PCAauto | XGB] PR=0.5368 time=11.1s
[Mol2Vec+Phys_PCAauto | LGB] PR=0.5298 time=13.8s
[MorganFP_PCA20       | XGB] PR=0.4996 time=3.5s
[MorganFP_PCA20     

In [ ]:
def per_target_full_benchmark():
    rows = []

    for feat_name, (X_tr, X_te) in representations.items():
        for model_name, model_fn in models:

            print(f"\nRunning {feat_name} | {model_name}")

            scores = []

            for t in range(Y.shape[1]):
                pr, roc = eval_task(t, model_fn, X_tr, X_te)

                rows.append({
                    "target": target_cols[t],
                    "features": feat_name,
                    "model": model_name,
                    "PR-AUC": pr,
                    "ROC-AUC": roc})

    df = pd.DataFrame(rows)
    return df

df_targets = per_target_full_benchmark()
df_targets.to_csv(BASE + "per_target_full.csv", index=False)

print(df_targets.head())


Running PhysChem | RF

Running PhysChem | ExtraTrees

Running PhysChem | XGB

Running PhysChem | LGB

Running PhysChem | LogReg

Running PhysChem | KNN

Running Mol2Vec+Phys | RF

Running Mol2Vec+Phys | ExtraTrees

Running Mol2Vec+Phys | XGB

Running Mol2Vec+Phys | LGB

Running Mol2Vec+Phys | LogReg

Running Mol2Vec+Phys | KNN

Running MorganFP | RF

Running MorganFP | ExtraTrees

Running MorganFP | XGB

Running MorganFP | LGB

Running MorganFP | LogReg

Running MorganFP | KNN

Running ALL | RF

Running ALL | ExtraTrees

Running ALL | XGB

Running ALL | LGB

Running ALL | LogReg

Running ALL | KNN
            target  features model    PR-AUC   ROC-AUC
0   acute_toxicity  PhysChem    RF  0.484904  0.624291
1  carcinogenicity  PhysChem    RF  0.785628  0.653509
2   cardiotoxicity  PhysChem    RF  0.196380  0.809463
3  dermal_toxicity  PhysChem    RF  0.599261  0.475709
4     genotoxicity  PhysChem    RF  0.582861  0.702697


In [ ]:
best_per_target = (
    df_targets
    .sort_values("PR-AUC", ascending=False)
    .groupby("target")
    .first()
    .reset_index())

print(best_per_target)
best_per_target.to_csv(BASE + "best_per_target.csv", index=False)

                     target  features       model    PR-AUC   ROC-AUC
0            acute_toxicity  PhysChem          RF  0.484904  0.624291
1           carcinogenicity  PhysChem          RF  0.785628  0.653509
2            cardiotoxicity  PhysChem      LogReg  0.251334  0.794492
3           dermal_toxicity  PhysChem         XGB  0.667188  0.570850
4   endocrine_metabolic_tox  PhysChem          RF  0.280943  0.558400
5              genotoxicity       ALL         LGB  0.689340  0.785184
6            hepatotoxicity  MorganFP          RF  0.789497  0.676923
7     immuno_hematotoxicity  PhysChem  ExtraTrees  0.560581  0.673489
8    neuro_sensory_toxicity       ALL         XGB  1.000000  1.000000
9           ocular_toxicity  PhysChem  ExtraTrees  0.994192  0.878761
10         oxidative_stress  MorganFP  ExtraTrees  0.258511  0.602605
11      reprod_dev_toxicity  PhysChem          RF       NaN       NaN
12     respiratory_toxicity  PhysChem  ExtraTrees  0.924803  0.803922


In [ ]:
pivot = df_targets.pivot_table(
    index="target",
    columns=["features", "model"],
    values="PR-AUC")

print(pivot.round(3))

features                       ALL                                     \
model                   ExtraTrees    KNN    LGB LogReg     RF    XGB   
target                                                                  
acute_toxicity               0.327  0.247  0.402  0.452  0.328  0.351   
carcinogenicity              0.656  0.614  0.613  0.745  0.616  0.589   
cardiotoxicity               0.223  0.095  0.230  0.239  0.209  0.244   
dermal_toxicity              0.526  0.593  0.578  0.596  0.547  0.563   
endocrine_metabolic_tox      0.244  0.205  0.229  0.211  0.221  0.239   
genotoxicity                 0.649  0.601  0.689  0.628  0.655  0.653   
hepatotoxicity               0.668  0.664  0.485  0.665  0.625  0.527   
immuno_hematotoxicity        0.318  0.417  0.277  0.381  0.376  0.303   
neuro_sensory_toxicity       0.858  0.824  0.824  0.980  0.953  1.000   
ocular_toxicity              0.979  0.972  0.983  0.989  0.978  0.979   
oxidative_stress             0.180  0.168  0.169  0

In [ ]:
best_models = {}

for _, row in best_per_target.iterrows():

    target = row["target"]
    feat = row["features"]
    model_name = row["model"]

    X_tr, X_te = representations[feat]

    model_fn = dict(models)[model_name]

    best_models[target] = (model_fn, X_tr, X_te)

def ensemble_score(best_models):
    pr_list = []
    roc_list = []

    for t, target in enumerate(target_cols):

        if target not in best_models:
            continue

        model_fn, X_tr, X_te = best_models[target]

        pr, roc = eval_task(t, model_fn, X_tr, X_te)

        pr_list.append(pr)
        roc_list.append(roc)

    return np.nanmean(pr_list), np.nanmean(roc_list)

pr, roc = ensemble_score(best_models)
print(f"\n FINAL ENSEMBLE → PR-AUC={pr:.4f}, ROC-AUC={roc:.4f}")


 FINAL ENSEMBLE → PR-AUC=0.6405, ROC-AUC=0.7185


Анализ целевых категорий показал неоднородность в задаче токсичности. Для простых категорий ocular_toxicity, neuro_sensory_toxicity, respiratory_toxicity большинство моделей показывают высокие PR-AUC = 0.9–1.0. Для сложных oxidative_stress, cardiotoxicity, endocrine_metabolic_tox качество остается низким. Лучшие результаты дают простые ансамблевые методы Random Forest, ExtraTrees с PhysChem признаками, а не бустинги. Добавление дополнительных признаков Mol2Vec, MorganFP, ALL часто ухудшает результаты. Мультитаргетная задача сводится к набору независимых бинарных задач, что обосновывает стратегию выбора лучшей модели на таргет и их объединение в ансамбль. Итоговый ансамбль показал высокое качество PR-AUC ~0.64.

In [ ]:
def inspect_targets():

    rows = []

    for t, name in enumerate(target_cols):

        y_tr = Y_train[:, t]
        y_te = Y_test[:, t]

        m_tr = ~np.isnan(y_tr)
        m_te = ~np.isnan(y_te)

        y_tr_clean = y_tr[m_tr]
        y_te_clean = y_te[m_te]

        row = {
            "target": name,

            "train_n": len(y_tr_clean),
            "test_n": len(y_te_clean),

            "train_pos": int(np.sum(y_tr_clean == 1)),
            "train_neg": int(np.sum(y_tr_clean == 0)),

            "test_pos": int(np.sum(y_te_clean == 1)),
            "test_neg": int(np.sum(y_te_clean == 0)),

            "train_unique": len(np.unique(y_tr_clean)),
            "test_unique": len(np.unique(y_te_clean)),}

        rows.append(row)

    df = pd.DataFrame(rows)
    return df


df_check = inspect_targets()
print(df_check.to_string(index=False))

                 target  train_n  test_n  train_pos  train_neg  test_pos  test_neg  train_unique  test_unique
         acute_toxicity       93     124         30         63        30        94             2            2
        carcinogenicity       22      31         14          8        19        12             2            2
         cardiotoxicity     7781    1726        574       7207        77      1649             2            2
        dermal_toxicity       29      45         25          4        26        19             2            2
           genotoxicity      157     200         73         84        71       129             2            2
         hepatotoxicity       67      23         35         32        13        10             2            2
        ocular_toxicity       43     118         25         18       113         5             2            2
       oxidative_stress       51     118          9         42        19        99             2            2
   respira

In [ ]:
bad_targets = df_check[
    (df_check["train_n"] < 20) |
    (df_check["train_unique"] < 2) |
    (df_check["test_unique"] < 2)
]

print("\nPROBLEMATIC TARGETS:")
print(bad_targets.to_string(index=False))


PROBLEMATIC TARGETS:
             target  train_n  test_n  train_pos  train_neg  test_pos  test_neg  train_unique  test_unique
reprod_dev_toxicity       18      15         16          2        13         2             2            2


In [ ]:
df_check["train_ratio"] = df_check["train_pos"] / (df_check["train_n"] + 1e-6)
df_check["test_ratio"] = df_check["test_pos"] / (df_check["test_n"] + 1e-6)

print(df_check[["target", "train_ratio", "test_ratio"]].round(3))

                     target  train_ratio  test_ratio
0            acute_toxicity        0.323       0.242
1           carcinogenicity        0.636       0.613
2            cardiotoxicity        0.074       0.045
3           dermal_toxicity        0.862       0.578
4              genotoxicity        0.465       0.355
5            hepatotoxicity        0.522       0.565
6           ocular_toxicity        0.581       0.958
7          oxidative_stress        0.176       0.161
8      respiratory_toxicity        0.771       0.739
9    neuro_sensory_toxicity        0.955       0.824
10    immuno_hematotoxicity        0.405       0.319
11      reprod_dev_toxicity        0.889       0.867
12  endocrine_metabolic_tox        0.465       0.200


In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import time
# import numpy as np
# import pandas as pd

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.metrics import average_precision_score, roc_auc_score

# from google.colab import drive
# drive.mount('/content/drive')

# BASE = '/content/drive/MyDrive/Colab Notebooks/'

# data = np.load(BASE + 'aligned_data.npz', allow_pickle=True)

# X_phys = data['X_phys']
# X_fp   = data['X_fp']
# X_m2v  = data['X_m2v']
# Y      = data['Y']

# targets_df = pd.read_csv(BASE + '2_targets_only.csv')
# target_cols = [c for c in targets_df.columns if c != 'smiles']

# print("Shapes:", X_phys.shape, X_fp.shape, X_m2v.shape, Y.shape)


# idx = np.arange(len(Y))
# train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

# X_phys_tr, X_phys_te = X_phys[train_idx], X_phys[test_idx]
# X_fp_tr,   X_fp_te   = X_fp[train_idx],   X_fp[test_idx]
# X_m2v_tr,  X_m2v_te  = X_m2v[train_idx],  X_m2v[test_idx]

# Y_train, Y_test = Y[train_idx], Y[test_idx]

# print("Train/Test:", len(train_idx), len(test_idx))


# representations = {
#     "PhysChem": (X_phys_tr, X_phys_te),

#     "MorganFP": (X_fp_tr, X_fp_te),

#     "Mol2Vec+Phys": (
#         np.hstack([X_phys_tr, X_m2v_tr]),
#         np.hstack([X_phys_te, X_m2v_te])
#     ),

#     "ALL": (
#         np.hstack([X_phys_tr, X_fp_tr, X_m2v_tr]),
#         np.hstack([X_phys_te, X_fp_te, X_m2v_te])
#     )}

# for k, (X_tr, X_te) in representations.items():
#     print(k, X_tr.shape)


# def pr_auc(y_true, y_prob):
#     mask = ~np.isnan(y_true)
#     if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
#         return np.nan
#     return average_precision_score(y_true[mask], y_prob[mask])

# def roc_auc(y_true, y_prob):
#     mask = ~np.isnan(y_true)
#     if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
#         return np.nan
#     return roc_auc_score(y_true[mask], y_prob[mask])


# def rf():
#     return RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)

# def et():
#     return ExtraTreesClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)

# def xgb():
#     from xgboost import XGBClassifier
#     return XGBClassifier(
#         n_estimators=400,
#         max_depth=5,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         scale_pos_weight=5,
#         eval_metric="logloss",
#         n_jobs=-1,
#         random_state=42
#     )

# def lgb():
#     from lightgbm import LGBMClassifier
#     return LGBMClassifier(
#         n_estimators=400,
#         num_leaves=63,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         class_weight="balanced",
#         n_jobs=-1,
#         random_state=42,
#         verbose=-1)

# def logreg():
#     return LogisticRegression(max_iter=1000, class_weight="balanced", solver='saga', n_jobs=-1)

# def knn():
#     return KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

# models = [
#     ("RF", rf),
#     ("ExtraTrees", et),
#     ("XGB", xgb),
#     ("LGB", lgb),
#     ("LogReg", logreg),
#     ("KNN", knn)
# ]


# def eval_task(task_i, model_fn, X_tr, X_te):

#     y_tr = Y_train[:, task_i]
#     y_te = Y_test[:, task_i]

#     m_tr = ~np.isnan(y_tr)
#     m_te = ~np.isnan(y_te)

#     if m_tr.sum() < 20 or len(np.unique(y_tr[m_tr])) < 2:
#         return np.nan, np.nan

#     model = model_fn()
#     model.fit(X_tr[m_tr], y_tr[m_tr])

#     prob = model.predict_proba(X_te[m_te])[:, 1]

#     return pr_auc(y_te[m_te], prob), roc_auc(y_te[m_te], prob)


# rows = []

# for feat_name, (X_tr, X_te) in representations.items():
#     for model_name, model_fn in models:

#         print(f"\nRunning {feat_name} | {model_name}")

#         for t in range(Y.shape[1]):

#             pr, roc = eval_task(t, model_fn, X_tr, X_te)

#             rows.append({
#                 "target": target_cols[t],
#                 "features": feat_name,
#                 "model": model_name,
#                 "PR-AUC": pr,
#                 "ROC-AUC": roc
#             })


# df_targets = pd.DataFrame(rows)
# df_targets.to_csv(BASE + "per_target_random_split.csv", index=False)

# print("\nDONE")
# print(df_targets.head())


# best_per_target = (
#     df_targets
#     .sort_values("PR-AUC", ascending=False)
#     .groupby("target")
#     .first()
#     .reset_index())

# print("\nBEST PER TARGET:")
# print(best_per_target)

# best_per_target.to_csv(BASE + "best_per_target_random.csv", index=False)

# best_models = {}

# for _, row in best_per_target.iterrows():

#     target = row["target"]
#     feat = row["features"]
#     model_name = row["model"]

#     X_tr, X_te = representations[feat]
#     model_fn = dict(models)[model_name]

#     best_models[target] = (model_fn, X_tr, X_te)


# def ensemble_score():
#     pr_list, roc_list = [], []

#     for t, target in enumerate(target_cols):

#         if target not in best_models:
#             continue

#         model_fn, X_tr, X_te = best_models[target]

#         pr, roc = eval_task(t, model_fn, X_tr, X_te)

#         pr_list.append(pr)
#         roc_list.append(roc)

#     return np.nanmean(pr_list), np.nanmean(roc_list)


# pr, roc = ensemble_score()
# print(f"\nFINAL ENSEMBLE → PR-AUC={pr:.4f}, ROC-AUC={roc:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shapes: (10000, 78) (10000, 2048) (10000, 300) (10000, 13)
Train/Test: 8000 2000
PhysChem (8000, 78)
MorganFP (8000, 2048)
Mol2Vec+Phys (8000, 378)
ALL (8000, 2426)

Running PhysChem | RF

Running PhysChem | ExtraTrees

Running PhysChem | XGB

Running PhysChem | LGB

Running PhysChem | LogReg

Running PhysChem | KNN

Running MorganFP | RF

Running MorganFP | ExtraTrees

Running MorganFP | XGB

Running MorganFP | LGB

Running MorganFP | LogReg

Running MorganFP | KNN

Running Mol2Vec+Phys | RF

Running Mol2Vec+Phys | ExtraTrees

Running Mol2Vec+Phys | XGB

Running Mol2Vec+Phys | LGB

Running Mol2Vec+Phys | LogReg

Running Mol2Vec+Phys | KNN

Running ALL | RF

Running ALL | ExtraTrees

Running ALL | XGB

Running ALL | LGB

Running ALL | LogReg

Running ALL | KNN

DONE
            target  features model    PR-AUC   ROC-AUC
0   acute_toxicity  PhysChem    RF  0.4